# Getting Started with Strands Agents

This workshop introduces Strands Agents, a powerful framework for building AI agents with tool integration capabilities. You'll learn the fundamentals of creating agents, using built-in tools, and developing custom tools.

## Overview

In this lab, you will:
- Understand the core concepts of Strands Agents
- Create your first AI agent with built-in tools
- Build custom tools for specific use cases
- Explore conversation history and agent memory
- Learn best practices for agent development

## Prerequisites

Before starting this lab, ensure you have:
- AWS credentials configured (IAM role or environment variables)
- Required Python packages installed
- Nova Pro model ID based on AWS region

If you're not running in an environment with an IAM role assumed, set your AWS credentials as environment variables:

In [6]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>


os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
os.environ["AWS_PROFILE"] = "sso-genai"
import sys
print(sys.executable)

/Users/rafap/Library/CloudStorage/OneDrive-Personal/101.aws_courses/01.GenAI.AWS.Courses/GenAiDemos/02.agentcore.demos/.venv/bin/python


In [7]:
!aws sts get-caller-identity

Account: '730558614357'
Arn: arn:aws:sts::730558614357:assumed-role/AWSReservedSSO_acme_admin_0a6a5b422ccc0864/rafa
UserId: AROA2UGFPZNK7EOSDSIJ5:rafa


Install required packages for Strands Agents:

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich

Setup Nova Pro model ID based on AWS region:

⚠️ **Important**: If you haven't enabled Nova Pro model access yet, go to the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/home#/modelaccess) to enable it.

In [8]:
import boto3

region = boto3.session.Session().region_name
print(f"Region: {region}")
NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: us-east-1
Nova Pro Model ID: us.amazon.nova-pro-v1:0


## What are Strands Agents?

Strands Agents is a Python framework that simplifies the creation of AI agents with tool integration capabilities. Key features include:

- **Simple Agent Creation**: Easy-to-use API for creating AI agents with minimal code
- **Built-in Tools**: Pre-built tools like calculators, web search, and more
- **Custom Tool Support**: Create your own tools with simple Python functions
- **Conversation Memory**: Automatic conversation history management
- **Model Flexibility**: Support for various language models including models in Amazon Bedrock and OpenAI

Strands Agents provides a foundation for building sophisticated AI applications that can interact with external systems and perform complex tasks.

## Creating Your First Strands Agent

Let's start by creating a simple agent with a built-in calculator tool. This demonstrates the basic structure of Strands Agents and how tools are integrated:

In [9]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a helpful assistant that provides concise responses.",
    tools=[calculator],
)

agent("What is the area of a circle with radius 8.26cm?")

<thinking> To find the area of a circle, we use the formula A = πr², where r is the radius. In this case, the radius is 8.26 cm. I will

DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the bash tool vended by strands-agents (from strands.vended_tools import bash). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while bash executes arbitrary commands, so review it against your threat model before switching.


 use the calculator tool in evaluate mode to compute the area. </thinking>

Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ pi * 8.26**2        │                                                                            │
│  │ Result    │ 214.3433269321      │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The area of the circle with a radius of 8.26 cm is approximately 214.34 cm².

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The area of the circle with a radius of 8.26 cm is approximately 214.34 cm².'}], 'metadata': {'usage': {'inputTokens': 1721, 'outputTokens': 27, 'totalTokens': 1748}, 'metrics': {'latencyMs': 590, 'timeToFirstByteMs': 565}}, 'tracking_id': '5d44dc91-1444-4db4-a889-e43accbf0811'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_kQeXUEUWJJh2rQP9gc0XfI', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': 'pi * 8.26**2'}}, call_count=1, success_count=1, error_count=0, total_time=0.0302731990814209)}, cycle_durations=[2.16506290435791, 0.7187409400939941], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='8683b69a-e36b-4c6f-b6bd-d0cf561f5740', usage={'inputTokens': 1593, 'outputTokens': 85, 'totalTokens': 1678}), EventLoopCycleMetric(event_loop_cycle_id='27b5df4e-b414-4a3d-a187-2e974b7bfdc0', usa

## Building Custom Tools

One of the powerful features of Strands Agents is the ability to create custom tools. Let's create a weather tool and combine it with the built-in calculator:

In [10]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

# Create a custom weather tool for demonstration
@tool
def weather(city: str) -> str:
    """Get weather information for a city
    Args:
        city: City or location name
    """
    return f"Weather for {city}: Sunny, 35°C" # dummy result for demo purpose

# Create your first agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a helpful assistant that provides concise responses.",
    tools=[weather, calculator],
)

agent("How is the weather in HK? Return temperature in Fahrenheit")

<thinking> I need to get the weather information for Hong Kong and convert the temperature from Celsius to Fahrenheit. </thinking>

Tool #1: weather
<thinking> I have the weather information for Hong Kong. Now, I need to convert the temperature from Celsius to Fahrenheit. The formula for conversion is F = (C * 9/5) + 3

DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the bash tool vended by strands-agents (from strands.vended_tools import bash). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while bash executes arbitrary commands, so review it against your threat model before switching.


2. </thinking> 
Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ (35 * 9/5) + 32     │                                                                            │
│  │ Result    │ 95                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The current temperature in Hong Kong is 95°F.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The current temperature in Hong Kong is 95°F.'}], 'metadata': {'usage': {'inputTokens': 1819, 'outputTokens': 14, 'totalTokens': 1833}, 'metrics': {'latencyMs': 506, 'timeToFirstByteMs': 548}}, 'tracking_id': '1bce930a-6b90-48e6-b2ef-b548c991edbd'}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'weather': ToolMetrics(tool={'toolUseId': 'tooluse_u0zF0gDRMyn750BGr9ZM7j', 'name': 'weather', 'input': {'city': 'Hong Kong'}}, call_count=1, success_count=1, error_count=0, total_time=0.0009641647338867188), 'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_3Q6exkknPt34wApUzSA5af', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '(35 * 9/5) + 32'}}, call_count=1, success_count=1, error_count=0, total_time=0.006884098052978516)}, cycle_durations=[2.145479917526245, 1.070847988128662, 0.643273115158081], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cy

## Understanding Agent Execution Flow

Let's examine how Strands Agents process requests with an agent loop. This helps understand the internal workings of the agent framework:

In [14]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="white")
table.add_column("Tool Name", style="yellow")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 3

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                        ┃ Tool Name  ┃ Tool Input                  ┃ Tool Result                ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ How is the weather in HK?   │            │                             │                            │
│           │ Return temperature in       │            │                             │                            │
│           │ Fahrenheit                  │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I need to get    │ weather    │ {                           │                            │
│           │ the weather information for │            │   "city": "Hong Kong"       │                            │
│           │ Hong Kong and convert the   │            │ }                           │                            │
│           │ temperature from Celsius to │            │                             │                            │
│           │ Fahrenheit. </thinking>     │            │                             │                            │
│           │                             │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Weather for     │
│           │                             │            │                             │ Hong Kong: Sunny,          │
│           │                             │            │                             │ 35\u00b0C"                 │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I have the       │ calculator │ {                           │                            │
│           │ weather information for     │            │   "mode": "evaluate",       │                            │
│           │ Hong Kong. Now, I need to   │            │   "expression": "(35 * 9/5) │                            │
│           │ convert the temperature     │            │ + 32"                       │                            │
│           │ from Celsius to Fahrenheit. │            │ }                           │                            │
│           │ The formula for conversion  │            │                             │                            │
│           │ is F = (C * 9/5) + 32.      │            │                             │                            │
│           │ </thinking>                 │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Result: 95"     │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ The current temperature in  │            │

## Agent Loop

The agent loop is a core concept in the Strands Agents SDK that enables intelligent, autonomous behavior through a cycle of reasoning, tool use, and response generation. 

![strands-agents-agent-loop](images/strands-agents-agent-loop.png)

At its core, the agent loop follows these steps:

1. Receives user input and contextual information
2. Processes the input using a language model (LLM)
3. Decides whether to use tools to gather information or perform actions
4. Executes tools and receives results
5. Continues reasoning with the new information
6. Produces a final response or iterates again through the loop
   
This cycle may repeat multiple times within a single user interaction, allowing the agent to perform complex, multi-step reasoning and autonomous behavior.

Reference: [Strands Agents - Agent Loop](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/agent-loop/)


## Strands Agents Conversation Management

Strands Agents includes built-in conversation management with a default `SlidingWindowConversationManager` strategy. This system handles context management, memory optimization, and conversation flow to ensure agents can maintain coherent long-term interactions while respecting model context limits. This automatically maintains conversation context within sessions.

**Reference:** [Strands Agents - Conversation Management](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/conversation-management/)

In [15]:
agent("What did we talk about?")

We discussed the weather in Hong Kong. The current temperature is 95°F.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'We discussed the weather in Hong Kong. The current temperature is 95°F.'}], 'metadata': {'usage': {'inputTokens': 1843, 'outputTokens': 19, 'totalTokens': 1862}, 'metrics': {'latencyMs': 591, 'timeToFirstByteMs': 608}}, 'tracking_id': '2240f1de-0ad7-4a4b-9b15-897155f612e2'}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'weather': ToolMetrics(tool={'toolUseId': 'tooluse_u0zF0gDRMyn750BGr9ZM7j', 'name': 'weather', 'input': {'city': 'Hong Kong'}}, call_count=1, success_count=1, error_count=0, total_time=0.0009641647338867188), 'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_3Q6exkknPt34wApUzSA5af', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '(35 * 9/5) + 32'}}, call_count=1, success_count=1, error_count=0, total_time=0.006884098052978516)}, cycle_durations=[2.145479917526245, 1.070847988128662, 0.643273115158081, 0.736731767654419], agent_invocations=[AgentInvocati

Let's examine how Strands Agents manage conversation history. This helps understand the internal workings of the agent framework:

In [16]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="white")
table.add_column("Tool Name", style="yellow")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 4

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                        ┃ Tool Name  ┃ Tool Input                  ┃ Tool Result                ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ How is the weather in HK?   │            │                             │                            │
│           │ Return temperature in       │            │                             │                            │
│           │ Fahrenheit                  │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I need to get    │ weather    │ {                           │                            │
│           │ the weather information for │            │   "city": "Hong Kong"       │                            │
│           │ Hong Kong and convert the   │            │ }                           │                            │
│           │ temperature from Celsius to │            │                             │                            │
│           │ Fahrenheit. </thinking>     │            │                             │                            │
│           │                             │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Weather for     │
│           │                             │            │                             │ Hong Kong: Sunny,          │
│           │                             │            │                             │ 35\u00b0C"                 │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ <thinking> I have the       │ calculator │ {                           │                            │
│           │ weather information for     │            │   "mode": "evaluate",       │                            │
│           │ Hong Kong. Now, I need to   │            │   "expression": "(35 * 9/5) │                            │
│           │ convert the temperature     │            │ + 32"                       │                            │
│           │ from Celsius to Fahrenheit. │            │ }                           │                            │
│           │ The formula for conversion  │            │                             │                            │
│           │ is F = (C * 9/5) + 32.      │            │                             │                            │
│           │ </thinking>                 │            │                             │                            │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ user      │                             │            │                             │ {                          │
│           │                             │            │                             │   "text": "Result: 95"     │
│           │                             │            │                             │ }                          │
├───────────┼─────────────────────────────┼────────────┼─────────────────────────────┼────────────────────────────┤
│ assistant │ The current temperature in  │            │

### Examine Strands Agents Conversation Management in a New Session

Strands Agents Conversation Management maintains the conversation history by accumulating the user and agent messages in input prompt within a session. The conversation history is stored in memory and is volatile when the session is end. Let's simulate a new session by initialising a new agent and test the Strands Agents Conversation Management behavior.

In [17]:
# Initialise a new agent for a new session
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="You are a helpful assistant that provides concise responses.",
    tools=[weather, calculator],
)

agent("What did we talk about?")

<thinking> I need to review the conversation history to provide a concise summary of what we talked about. </thinking>



AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '<thinking> I need to review the conversation history to provide a concise summary of what we talked about. </thinking>\n\n'}], 'metadata': {'usage': {'inputTokens': 1628, 'outputTokens': 34, 'totalTokens': 1662}, 'metrics': {'latencyMs': 670, 'timeToFirstByteMs': 1584}}, 'tracking_id': 'ce45ff32-9a66-4027-87b2-1c7f2e2daaa5'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[1.8306059837341309], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='a8d4a663-666c-4838-ad65-40ae0385f764', usage={'inputTokens': 1628, 'outputTokens': 34, 'totalTokens': 1662})], usage={'inputTokens': 1628, 'outputTokens': 34, 'totalTokens': 1662})], traces=[<strands.telemetry.metrics.Trace object at 0x112d2cd60>], accumulated_usage={'inputTokens': 1628, 'outputTokens': 34, 'totalTokens': 1662}, accumulated_metrics={'latencyMs': 670}), state={}, interrupts=None, struc

 You will see the agent forget the past conversation about Hong Kong's weather enquiry. This demonstrates the Strands Agents built-in conversation management is Short-Term Memory only. To enable Long-Term Memory across sessions and agents, you will explore Bedrock AgentCore Memory as a managed agent memory service in [Lab 6: Strands Agents with Bedrock AgentCore Memory](../06-bedrock-agentcore-memory/06-agentcore-memory.ipynb) later.

## Summary

In this introduction to Strands Agents, you've learned:

### What We Accomplished
- **Created your first AI agent** with built-in tools (calculator)
- **Built custom tools** using the `@tool` decorator for specific use cases
- **Explored conversation management** with different ConversationManager types
- **Examined agent execution flow** and message history tracking

### Additional Strands Capabilities
Beyond what we've covered, Strands Agents offers:

- **Model Context Protocol (MCP) Support**: Integrate with MCP servers for extended tool capabilities
- **Multi-Agent Patterns**: Coordinate multiple agents for complex workflows
- **Session Management**: Persistent conversation storage and retrieval
- **Streaming Responses**: Real-time response generation for better user experience
- **Custom Model Integration**: Support for various LLM providers beyond Amazon Bedrock

### Next Steps: Bedrock AgentCore Integration

Now that you understand the fundamentals of Strands Agents, we'll explore how to enhance these capabilities by integrating with **Amazon Bedrock AgentCore**. This integration provides:

- **Code Interpreter**: Execute Python code dynamically within agents
- **Browser Automation**: Web interaction and data extraction capabilities  
- **Secure Credential Management**: Safe handling of external API keys and secrets
- **Runtime Deployment**: Scalable agent deployment in cloud environments
- **Enhanced Observability**: Monitoring and debugging tools for production agents